In [1]:
# ============================================================
# PS S6E5 - exp_20260504_010_blend_core_min
#
# Compare:
# - Avg
# - Ridge
# - ElasticNet
# - LogisticRegression
# - Hill Climb
# - Nelder-Mead
#
# ============================================================

In [2]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.stats import rankdata, spearmanr

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet, LogisticRegression

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260504_010_blend_core_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    COMP_BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = COMP_BASE / "train.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    NPY_BASE = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    ID_COL = "id"
    TARGET = "PitNextLap"

    SEED = 42
    N_SPLITS = 5

    EPS = 1e-6


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def safe_clip(x, eps=CFG.EPS):
    return np.clip(np.asarray(x, dtype=float), eps, 1.0 - eps)


def safe_logit(p, eps=CFG.EPS):
    p = safe_clip(p, eps)
    return np.log(p / (1.0 - p))


def rank01(x):
    x = np.asarray(x, dtype=float)
    return rankdata(x, method="average") / (len(x) + 1.0)


def softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()


def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    w = np.clip(w, 0.0, None)
    s = w.sum()
    if s <= 0:
        return np.ones_like(w) / len(w)
    return w / s


def weighted_average(X, w):
    w = normalize_weights(w)
    return np.asarray(X) @ w


def auc(y, pred):
    return roc_auc_score(y, pred)


def sanitize_name(name: str) -> str:
    return (
        name.replace(" ", "_")
            .replace("/", "_")
            .replace("+", "plus")
            .replace("-", "_")
            .replace(".", "_")
    )


seed_everything(CFG.SEED)

In [5]:
# ============================================================
# Load data
# ============================================================

train = pd.read_csv(CFG.TRAIN_PATH)
sub = pd.read_csv(CFG.SUB_PATH)

y = train[CFG.TARGET].astype(int).values

print("train:", train.shape)
print("sub  :", sub.shape)
print("target mean:", y.mean())

train: (439140, 16)
sub  : (188165, 2)
target mean: 0.19898210137996994


In [6]:
# ============================================================
# Load OOF / pred
# ============================================================

MODEL_SPECS = [
    {
        "key": "007",
        "name": "007_realmlp_shared001_min",
        "family": "RealMLP",
        "oof": "oof_exp_20260503_007_realmlp_shared001_min.npy",
        "pred": "pred_exp_20260503_007_realmlp_shared001_min.npy",
        "public_lb": 0.95273,
    },
    {
        "key": "008",
        "name": "008_realmlp_plus_orig_prior",
        "family": "RealMLP",
        "oof": "oof_exp_20260503_008_realmlp_shared001_plus_006b_orig_prior.npy",
        "pred": "pred_exp_20260503_008_realmlp_shared001_plus_006b_orig_prior.npy",
        "public_lb": 0.95249,
    },
    {
        "key": "009",
        "name": "009_lgb_shared003_full_fe_single",
        "family": "LightGBM",
        "oof": "oof_exp_20260503_009_lgb_shared003_full_fe_single_min.npy",
        "pred": "pred_exp_20260503_009_lgb_shared003_full_fe_single_min.npy",
        "public_lb": 0.95076,
    },
    {
        "key": "012",
        "name": "cat_shared004_full_fe_oof_min",
        "family": "CatBoost",
        "oof": "oof_exp_20260505_012_cat_shared004_full_fe_oof_min.npy",
        "pred": "pred_exp_20260505_012_cat_shared004_full_fe_oof_min.npy",
        "public_lb": 0.95231,
    },
    {
        "key": "013",
        "name": "cat_shared004_no_isoriginaldata_min",
        "family": "CatBoost",
        "oof": "oof_exp_20260505_013_cat_shared004_no_isoriginaldata_min.npy",
        "pred": "pred_exp_20260505_013_cat_shared004_no_isoriginaldata_min.npy",
        "public_lb": 0.95191,
    },
    {
        "key": "014",
        "name": "cat_shared004_no_tyreageratio_str_min",
        "family": "CatBoost",
        "oof": "oof_exp_20260508_014_cat_shared004_no_tyreageratio_str_min.npy",
        "pred": "pred_exp_20260508_014_cat_shared004_no_tyreageratio_str_min.npy",
        "public_lb": 0.95233,
    },
    {
        "key": "015",
        "name": "cat_shared004_no_race_year_groupstats_min",
        "family": "CatBoost",
        "oof": "oof_exp_20260508_015_cat_shared004_no_race_year_groupstats_min.npy",
        "pred": "pred_exp_20260508_015_cat_shared004_no_race_year_groupstats_min.npy",
        "public_lb": 0.95244,
    },
    {
        "key": "016",
        "name": "xgb_shared005_fe_te_seedens_min",
        "family": "XGBoost",
        "oof": "oof_exp_20260508_016_xgb_shared005_fe_te_seedens_min.npy",
        "pred": "pred_exp_20260508_016_xgb_shared005_fe_te_seedens_min.npy",
        "public_lb": 0.95164,
    },
    {
        "key": "017",
        "name": "xgb_shared005_no_pitstop_pairte_min",
        "family": "XGBoost",
        "oof": "oof_exp_20260509_017_xgb_shared005_no_pitstop_pairte_min.npy",
        "pred": "pred_exp_20260509_017_xgb_shared005_no_pitstop_pairte_min.npy",
        "public_lb": 0.95164,
    },
]

oofs = {}
preds = {}

for spec in MODEL_SPECS:
    oof_path = CFG.NPY_BASE / spec["oof"]
    pred_path = CFG.NPY_BASE / spec["pred"]

    assert oof_path.exists(), f"missing oof: {oof_path}"
    assert pred_path.exists(), f"missing pred: {pred_path}"

    oof = np.load(oof_path)
    pred = np.load(pred_path)

    assert len(oof) == len(train), (spec["key"], len(oof), len(train))
    assert len(pred) == len(sub), (spec["key"], len(pred), len(sub))
    assert np.isfinite(oof).all(), spec["key"]
    assert np.isfinite(pred).all(), spec["key"]

    oofs[spec["key"]] = oof.astype(float)
    preds[spec["key"]] = pred.astype(float)

model_keys = [s["key"] for s in MODEL_SPECS]
model_names = [s["name"] for s in MODEL_SPECS]

X_raw = np.column_stack([oofs[k] for k in model_keys])
T_raw = np.column_stack([preds[k] for k in model_keys])

print("X_raw:", X_raw.shape)
print("T_raw:", T_raw.shape)

X_raw: (439140, 9)
T_raw: (188165, 9)


In [7]:
# ============================================================
# Individual reports
# ============================================================

individual_rows = []

for i, spec in enumerate(MODEL_SPECS):
    pred_oof = X_raw[:, i]
    individual_rows.append({
        "key": spec["key"],
        "name": spec["name"],
        "family": spec["family"],
        "cv_auc": auc(y, pred_oof),
        "public_lb": spec["public_lb"],
        "oof_min": float(pred_oof.min()),
        "oof_max": float(pred_oof.max()),
        "pred_min": float(T_raw[:, i].min()),
        "pred_max": float(T_raw[:, i].max()),
    })

individual_df = pd.DataFrame(individual_rows).sort_values("cv_auc", ascending=False)
display(individual_df)

best_single_key = individual_df.iloc[0]["key"]
best_single_auc = individual_df.iloc[0]["cv_auc"]

print("best single:", best_single_key, best_single_auc)

,key,name,family,cv_auc,public_lb,oof_min,oof_max,pred_min,pred_max
0,007,007_realmlp_shared001_min,RealMLP,0.953270,0.95273,0.000016,0.998289,0.000027,0.997671
1,008,008_realmlp_plus_orig_prior,RealMLP,0.953039,0.95249,0.000008,0.999127,0.000016,0.998304
3,012,cat_shared004_full_fe_oof_min,CatBoost,0.952748,0.95231,0.000060,0.997159,0.000126,0.996450
5,014,cat_shared004_no_tyreageratio_str_min,CatBoost,0.952726,0.95233,0.000071,0.997215,0.000153,0.996355
6,015,cat_shared004_no_race_year_groupstats_min,CatBoost,0.952714,0.95244,0.000045,0.997058,0.000141,0.996636
4,013,cat_shared004_no_isoriginaldata_min,CatBoost,0.952348,0.95191,0.000033,0.998772,0.000112,0.998480
7,016,xgb_shared005_fe_te_seedens_min,XGBoost,0.951986,0.95164,0.000017,0.997698,0.000027,0.997298
8,017,xgb_shared005_no_pitstop_pairte_min,XGBoost,0.951939,0.95164,0.000013,0.997702,0.000024,0.997621
2,009,009_lgb_shared003_full_fe_single,LightGBM,0.951013,0.95076,0.000014,0.999195,0.000015,0.998507


best single: 007 0.9532701289015783


In [8]:
# ============================================================
# Correlation matrix
# ============================================================

pearson = pd.DataFrame(
    np.corrcoef(X_raw.T),
    index=model_keys,
    columns=model_keys,
)

spearman_mat = np.zeros((len(model_keys), len(model_keys)))

for i in range(len(model_keys)):
    for j in range(len(model_keys)):
        spearman_mat[i, j] = spearmanr(X_raw[:, i], X_raw[:, j]).correlation

spearman_df = pd.DataFrame(
    spearman_mat,
    index=model_keys,
    columns=model_keys,
)

print("Pearson correlation")
display(pearson)

print("Spearman correlation")
display(spearman_df)

pearson.to_csv(CFG.OUTDIR / "corr_pearson.csv")
spearman_df.to_csv(CFG.OUTDIR / "corr_spearman.csv")

Pearson correlation


,007,008,009,012,013,014,015,016,017
007,1.000000,0.994703,0.956694,0.962091,0.963146,0.962718,0.961711,0.981305,0.981032
008,0.994703,1.000000,0.957863,0.962886,0.964353,0.963450,0.962528,0.982705,0.982322
009,0.956694,0.957863,1.000000,0.985403,0.984236,0.984907,0.985260,0.959935,0.960479
012,0.962091,0.962886,0.985403,1.000000,0.996026,0.997266,0.997518,0.962554,0.962384
013,0.963146,0.964353,0.984236,0.996026,1.000000,0.995716,0.995707,0.963950,0.963834
014,0.962718,0.963450,0.984907,0.997266,0.995716,1.000000,0.996917,0.962898,0.962739
015,0.961711,0.962528,0.985260,0.997518,0.995707,0.996917,1.000000,0.962398,0.962237
016,0.981305,0.982705,0.959935,0.962554,0.963950,0.962898,0.962398,1.000000,0.998485
017,0.981032,0.982322,0.960479,0.962384,0.963834,0.962739,0.962237,0.998485,1.000000


Spearman correlation


,007,008,009,012,013,014,015,016,017
007,1.000000,0.991720,0.968295,0.977637,0.975753,0.976924,0.976733,0.974170,0.973720
008,0.991720,1.000000,0.968978,0.976294,0.975556,0.975724,0.975377,0.974543,0.974159
009,0.968295,0.968978,1.000000,0.974737,0.973302,0.974674,0.974203,0.971451,0.971942
012,0.977637,0.976294,0.974737,1.000000,0.994238,0.995504,0.995553,0.975739,0.975409
013,0.975753,0.975556,0.973302,0.994238,1.000000,0.993758,0.993507,0.974431,0.974037
014,0.976924,0.975724,0.974674,0.995504,0.993758,1.000000,0.994676,0.975612,0.975146
015,0.976733,0.975377,0.974203,0.995553,0.993507,0.994676,1.000000,0.975249,0.974943
016,0.974170,0.974543,0.971451,0.975739,0.974431,0.975612,0.975249,1.000000,0.998203
017,0.973720,0.974159,0.971942,0.975409,0.974037,0.975146,0.974943,0.998203,1.000000


In [9]:
# ============================================================
# Meta feature builders
# ============================================================

def build_meta_features(X, T):
    X = np.asarray(X, dtype=float)
    T = np.asarray(T, dtype=float)

    X_rank = np.column_stack([rank01(X[:, i]) for i in range(X.shape[1])])
    T_rank = np.column_stack([rank01(T[:, i]) for i in range(T.shape[1])])

    X_logit = np.column_stack([safe_logit(X[:, i]) for i in range(X.shape[1])])
    T_logit = np.column_stack([safe_logit(T[:, i]) for i in range(T.shape[1])])

    X_meta = np.column_stack([X, X_rank, X_logit])
    T_meta = np.column_stack([T, T_rank, T_logit])

    feature_names = (
        [f"raw_{k}" for k in model_keys] +
        [f"rank_{k}" for k in model_keys] +
        [f"logit_{k}" for k in model_keys]
    )

    return X_meta, T_meta, feature_names


X_meta, T_meta, meta_feature_names = build_meta_features(X_raw, T_raw)

print("X_meta:", X_meta.shape)
print("T_meta:", T_meta.shape)
print("meta features:", meta_feature_names)

X_meta: (439140, 27)
T_meta: (188165, 27)
meta features: ['raw_007', 'raw_008', 'raw_009', 'raw_012', 'raw_013', 'raw_014', 'raw_015', 'raw_016', 'raw_017', 'rank_007', 'rank_008', 'rank_009', 'rank_012', 'rank_013', 'rank_014', 'rank_015', 'rank_016', 'rank_017', 'logit_007', 'logit_008', 'logit_009', 'logit_012', 'logit_013', 'logit_014', 'logit_015', 'logit_016', 'logit_017']


In [10]:
# ============================================================
# Save candidate helper
# ============================================================

candidate_records = {}
candidate_summary = []

def add_candidate(
    name,
    method,
    oof_pred,
    test_pred,
    weights=None,
    params=None,
    notes=None,
):
    oof_pred = np.asarray(oof_pred, dtype=float)
    test_pred = np.asarray(test_pred, dtype=float)

    score = auc(y, oof_pred)

    candidate_records[name] = {
        "method": method,
        "oof": oof_pred,
        "pred": test_pred,
        "score": float(score),
        "weights": None if weights is None else [float(x) for x in weights],
        "params": params or {},
        "notes": notes or "",
    }

    candidate_summary.append({
        "name": name,
        "method": method,
        "cv_auc": float(score),
        "delta_vs_007": float(score - auc(y, oofs["007"])),
        "delta_vs_best_single": float(score - best_single_auc),
        "weights": None if weights is None else json.dumps([round(float(x), 8) for x in weights]),
        "params": json.dumps(params or {}, ensure_ascii=False),
        "notes": notes or "",
        "oof_min": float(oof_pred.min()),
        "oof_max": float(oof_pred.max()),
        "pred_min": float(test_pred.min()),
        "pred_max": float(test_pred.max()),
    })

    print(f"{name}: {score:.9f}")

In [11]:
# ============================================================
# Simple averages
# ============================================================

print("\n" + "=" * 80)
print("Simple averages")
print("=" * 80)

avg_specs = {
    "avg_007_008_012_013_014_015_016_017": ["007", "008", "012", "013", "014", "015", "016", "017"],
    "avg_007_009_012_013_014_015_016_017": ["007", "009", "012", "013", "014", "015", "016", "017"],
    "avg_008_009_012_013_014_015_016_017": ["008", "009", "012", "013", "014", "015", "016", "017"],
    "avg_007_008_009_012_013_014_015_016_017": ["007", "008", "009", "012", "013", "014", "015", "016", "017"],
}

for name, keys in avg_specs.items():
    idx = [model_keys.index(k) for k in keys]
    w = np.zeros(len(model_keys))
    for j in idx:
        w[j] = 1.0 / len(idx)

    add_candidate(
        name=name,
        method="avg",
        oof_pred=weighted_average(X_raw, w),
        test_pred=weighted_average(T_raw, w),
        weights=w,
        notes=f"simple average of {keys}",
    )


Simple averages
avg_007_008_012_013_014_015_016_017: 0.954060056
avg_007_009_012_013_014_015_016_017: 0.953943685
avg_008_009_012_013_014_015_016_017: 0.953894808
avg_007_008_009_012_013_014_015_016_017: 0.954088373


In [12]:
# ============================================================
# Ridge / ElasticNet / LogisticRegression meta CV
# Two-stage search
# ============================================================

meta_folds = list(
    StratifiedKFold(
        n_splits=CFG.N_SPLITS,
        shuffle=True,
        random_state=CFG.SEED,
    ).split(X_meta, y)
)


def make_refined_grid_1d(best_value, factor_low=0.25, factor_high=4.0, n=17, min_value=1e-8):
    lo = max(best_value * factor_low, min_value)
    hi = max(best_value * factor_high, lo * 1.01)
    return np.geomspace(lo, hi, n).tolist()


def run_meta_regressor_cv(estimator_factory, params):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            estimator_factory(params),
        )
        model.fit(X_meta[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict(X_meta[va_idx])

    score = auc(y, oof_meta)
    return score, oof_meta


def run_meta_regressor_two_stage(name, estimator_factory, coarse_grid, refine_builder):
    history = []

    best = None

    print(f"\n{name} coarse search")
    for params in coarse_grid:
        score, oof_meta = run_meta_regressor_cv(estimator_factory, params)
        history.append({"stage": "coarse", "score": float(score), **params})
        print(params, score)

        if best is None or score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    refined_grid = refine_builder(best["params"])

    print(f"\n{name} refined search around {best['params']}")
    for params in refined_grid:
        score, oof_meta = run_meta_regressor_cv(estimator_factory, params)
        history.append({"stage": "refined", "score": float(score), **params})
        print(params, score)

        if score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    final_model = make_pipeline(
        StandardScaler(),
        estimator_factory(best["params"]),
    )
    final_model.fit(X_meta, y)
    test_meta = final_model.predict(T_meta)

    add_candidate(
        name=name,
        method=name.split("_")[0],
        oof_pred=best["oof"],
        test_pred=test_meta,
        params=best["params"],
        notes="two-stage meta CV on raw+rank+logit features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)

    print(f"\n{name} best:", best["params"], best["score"])
    display(hist_df.head(30))

    return best, hist_df


def run_meta_logreg_cv(params):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=params["C"],
                penalty="l2",
                solver="lbfgs",
                max_iter=3000,
                random_state=CFG.SEED,
            ),
        )
        model.fit(X_meta[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict_proba(X_meta[va_idx])[:, 1]

    score = auc(y, oof_meta)
    return score, oof_meta


def run_meta_logreg_two_stage(name, coarse_grid, refine_builder):
    history = []
    best = None

    print(f"\n{name} coarse search")
    for params in coarse_grid:
        score, oof_meta = run_meta_logreg_cv(params)
        history.append({"stage": "coarse", "score": float(score), **params})
        print(params, score)

        if best is None or score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    refined_grid = refine_builder(best["params"])

    print(f"\n{name} refined search around {best['params']}")
    for params in refined_grid:
        score, oof_meta = run_meta_logreg_cv(params)
        history.append({"stage": "refined", "score": float(score), **params})
        print(params, score)

        if score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    final_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=best["params"]["C"],
            penalty="l2",
            solver="lbfgs",
            max_iter=3000,
            random_state=CFG.SEED,
        ),
    )
    final_model.fit(X_meta, y)
    test_meta = final_model.predict_proba(T_meta)[:, 1]

    add_candidate(
        name=name,
        method="logreg",
        oof_pred=best["oof"],
        test_pred=test_meta,
        params=best["params"],
        notes="two-stage meta CV logistic regression on raw+rank+logit features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)

    print(f"\n{name} best:", best["params"], best["score"])
    display(hist_df.head(30))

    return best, hist_df


print("\n" + "=" * 80)
print("Ridge / ElasticNet / LogReg two-stage search")
print("=" * 80)


# ----------------------------
# Ridge two-stage
# ----------------------------

ridge_coarse_grid = [
    {"alpha": a}
    for a in np.geomspace(1e-3, 1e3, 19)
]

ridge_best, ridge_hist = run_meta_regressor_two_stage(
    name="ridge_meta_all",
    estimator_factory=lambda p: Ridge(
        alpha=p["alpha"],
        random_state=CFG.SEED,
    ),
    coarse_grid=ridge_coarse_grid,
    refine_builder=lambda best_p: [
        {"alpha": a}
        for a in make_refined_grid_1d(
            best_p["alpha"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ],
)


# ----------------------------
# ElasticNet two-stage
# ----------------------------

elastic_coarse_grid = [
    {"alpha": a, "l1_ratio": l1}
    for a in np.geomspace(1e-4, 1e-1, 7)
    for l1 in [0.05, 0.10, 0.20, 0.50, 0.90]
]


def build_elastic_refined_grid_fast(best_p):
    alpha_grid = make_refined_grid_1d(
        best_p["alpha"],
        factor_low=0.5,
        factor_high=2.0,
        n=7,
        min_value=1e-7,
    )

    l1_center = best_p["l1_ratio"]
    l1_grid = sorted(set([
        round(max(0.001, l1_center - 0.10), 6),
        round(l1_center, 6),
        round(min(0.999, l1_center + 0.10), 6),
        0.05,
        0.10,
        0.20,
        0.50,
        0.90,
    ]))

    return [
        {"alpha": a, "l1_ratio": l1}
        for a in alpha_grid
        for l1 in l1_grid
    ]


elastic_best, elastic_hist = run_meta_regressor_two_stage(
    name="elasticnet_meta_all",
    estimator_factory=lambda p: ElasticNet(
        alpha=p["alpha"],
        l1_ratio=p["l1_ratio"],
        max_iter=30000,
        random_state=CFG.SEED,
        selection="cyclic",
    ),
    coarse_grid=elastic_coarse_grid,
    refine_builder=build_elastic_refined_grid_fast,
)


# ----------------------------
# LogisticRegression two-stage
# ----------------------------

logreg_coarse_grid = [
    {"C": c}
    for c in np.geomspace(1e-3, 1e3, 19)
]

logreg_best, logreg_hist = run_meta_logreg_two_stage(
    name="logreg_meta_all",
    coarse_grid=logreg_coarse_grid,
    refine_builder=lambda best_p: [
        {"C": c}
        for c in make_refined_grid_1d(
            best_p["C"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ],
)


Ridge / ElasticNet / LogReg two-stage search

ridge_meta_all coarse search
{'alpha': np.float64(0.001)} 0.9536662829482623
{'alpha': np.float64(0.0021544346900318843)} 0.9536662834037386
{'alpha': np.float64(0.004641588833612777)} 0.9536662858112563
{'alpha': np.float64(0.01)} 0.9536662851931099
{'alpha': np.float64(0.021544346900318832)} 0.9536662833386707
{'alpha': np.float64(0.046415888336127774)} 0.953666285128042
{'alpha': np.float64(0.1)} 0.9536662871451513
{'alpha': np.float64(0.21544346900318823)} 0.9536662801503363
{'alpha': np.float64(0.46415888336127775)} 0.9536662795647238
{'alpha': np.float64(1.0)} 0.953666278230829
{'alpha': np.float64(2.154434690031882)} 0.9536662096816421
{'alpha': np.float64(4.6415888336127775)} 0.9536659100758229
{'alpha': np.float64(10.0)} 0.9536647316284359
{'alpha': np.float64(21.54434690031882)} 0.953661044189717
{'alpha': np.float64(46.41588833612773)} 0.9536518979973881
{'alpha': np.float64(100.0)} 0.9536308326078001
{'alpha': np.float64(215.44

,stage,score,alpha
30,refined,0.953666,0.117462
6,coarse,0.953666,0.100000
29,refined,0.953666,0.100000
28,refined,0.953666,0.085134
2,coarse,0.953666,0.004642
3,coarse,0.953666,0.010000
5,coarse,0.953666,0.046416
24,refined,0.953666,0.044721
22,refined,0.953666,0.032413
33,refined,0.953666,0.190365



elasticnet_meta_all coarse search
{'alpha': np.float64(0.0001), 'l1_ratio': 0.05} 0.9536404152765837
{'alpha': np.float64(0.0001), 'l1_ratio': 0.1} 0.9536187972966126
{'alpha': np.float64(0.0001), 'l1_ratio': 0.2} 0.953558064083538
{'alpha': np.float64(0.0001), 'l1_ratio': 0.5} 0.9533929572019366
{'alpha': np.float64(0.0001), 'l1_ratio': 0.9} 0.9534205646955718
{'alpha': np.float64(0.00031622776601683794), 'l1_ratio': 0.05} 0.9535467382743275
{'alpha': np.float64(0.00031622776601683794), 'l1_ratio': 0.1} 0.9534403697171026
{'alpha': np.float64(0.00031622776601683794), 'l1_ratio': 0.2} 0.9533427777006352
{'alpha': np.float64(0.00031622776601683794), 'l1_ratio': 0.5} 0.9535814060413965
{'alpha': np.float64(0.00031622776601683794), 'l1_ratio': 0.9} 0.9537533922747292
{'alpha': np.float64(0.001), 'l1_ratio': 0.05} 0.9533247920142136
{'alpha': np.float64(0.001), 'l1_ratio': 0.1} 0.9534836482640814
{'alpha': np.float64(0.001), 'l1_ratio': 0.2} 0.9536464108090418
{'alpha': np.float64(0.001),

,stage,score,alpha,l1_ratio
65,refined,0.954195,0.125992,0.10
30,coarse,0.954195,0.100000,0.05
57,refined,0.954195,0.100000,0.05
72,refined,0.954195,0.158740,0.10
73,refined,0.954194,0.158740,0.15
80,refined,0.954194,0.200000,0.15
66,refined,0.954193,0.125992,0.15
81,refined,0.954193,0.200000,0.20
31,coarse,0.954193,0.100000,0.10
58,refined,0.954193,0.100000,0.10



logreg_meta_all coarse search
{'C': np.float64(0.001)} 0.9542513979524647
{'C': np.float64(0.0021544346900318843)} 0.9542570451756709
{'C': np.float64(0.004641588833612777)} 0.95426473358349
{'C': np.float64(0.01)} 0.9542641834656924
{'C': np.float64(0.021544346900318832)} 0.9542626234918158
{'C': np.float64(0.046415888336127774)} 0.9542556807638075
{'C': np.float64(0.1)} 0.954261515382987
{'C': np.float64(0.21544346900318823)} 0.9542586080776093
{'C': np.float64(0.46415888336127775)} 0.9542583840483259
{'C': np.float64(1.0)} 0.9542586740240744
{'C': np.float64(2.154434690031882)} 0.954259274764812
{'C': np.float64(4.6415888336127775)} 0.954259257977256
{'C': np.float64(10.0)} 0.9542590054481682
{'C': np.float64(21.54434690031882)} 0.9542597820027672
{'C': np.float64(46.41588833612773)} 0.9542589406729283
{'C': np.float64(100.0)} 0.9542590917934658
{'C': np.float64(215.44346900318823)} 0.9542594038923498
{'C': np.float64(464.1588833612773)} 0.9542597251332945
{'C': np.float64(1000.0)}

,stage,score,C
35,refined,0.954266,0.012191
30,refined,0.954266,0.005452
36,refined,0.954265,0.014320
31,refined,0.954265,0.006404
34,refined,0.954265,0.010379
32,refined,0.954265,0.007522
2,coarse,0.954265,0.004642
29,refined,0.954265,0.004642
37,refined,0.954265,0.016821
33,refined,0.954265,0.008836


In [13]:
# ============================================================
# Hill Climb non-negative weights
# ============================================================

print("\n" + "=" * 80)
print("Hill Climb")
print("=" * 80)

def hill_climb_weights(X, y, init_candidates=None):
    n = X.shape[1]

    candidates = []

    # one-hot starts
    for i in range(n):
        w = np.zeros(n)
        w[i] = 1.0
        candidates.append(w)

    # avg start
    candidates.append(np.ones(n) / n)

    if init_candidates:
        for w in init_candidates:
            candidates.append(normalize_weights(w))

    best_w = None
    best_score = -np.inf

    for w in candidates:
        s = auc(y, weighted_average(X, w))
        if s > best_score:
            best_score = s
            best_w = normalize_weights(w)

    for step in [0.20, 0.10, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]:
        improved = True

        while improved:
            improved = False

            for i in range(n):
                for direction in [-1, 1]:
                    trial = best_w.copy()
                    trial[i] += direction * step
                    trial = normalize_weights(trial)

                    s = auc(y, weighted_average(X, trial))

                    if s > best_score + 1e-12:
                        best_score = s
                        best_w = trial
                        improved = True

    return best_w, best_score


hc_init = []

# Use candidates that already looked good as starts
for cand_name in ["avg_007_008_009", "avg_006b_007_008_009"]:
    if cand_name in candidate_records:
        hc_init.append(candidate_records[cand_name]["weights"])

hc_w, hc_score = hill_climb_weights(X_raw, y, init_candidates=hc_init)

add_candidate(
    name="hc_nonnegative_raw",
    method="hc",
    oof_pred=weighted_average(X_raw, hc_w),
    test_pred=weighted_average(T_raw, hc_w),
    weights=hc_w,
    params={"constraint": "nonnegative_sum1"},
    notes="greedy hill climb on raw OOF predictions; in-sample optimizer, interpret cautiously",
)

print("HC weights:")
for k, w in zip(model_keys, hc_w):
    print(k, w)


Hill Climb
hc_nonnegative_raw: 0.954293960
HC weights:
007 0.3517305707690582
008 0.13909412561480186
009 0.08873627797851072
012 0.04401607231747298
013 0.023669974597674743
014 0.11079880448054322
015 0.07439187386750544
016 0.1090237793722164
017 0.058538521002216314


In [14]:
# ============================================================
# Nelder-Mead softmax weights
# ============================================================

print("\n" + "=" * 80)
print("Nelder-Mead")
print("=" * 80)

def nm_optimize_weights(X, y, start_w=None, maxiter=500):
    n = X.shape[1]

    if start_w is None:
        start_w = np.ones(n) / n

    start_w = normalize_weights(start_w)
    z0 = np.log(np.clip(start_w, 1e-8, 1.0))

    def objective(z):
        w = softmax(z)
        p = weighted_average(X, w)
        return -auc(y, p)

    res = minimize(
        objective,
        z0,
        method="Nelder-Mead",
        options={
            "maxiter": maxiter,
            "xatol": 1e-7,
            "fatol": 1e-10,
            "disp": True,
        },
    )

    w = softmax(res.x)
    score = auc(y, weighted_average(X, w))

    return w, score, res


nm_w, nm_score, nm_res = nm_optimize_weights(
    X_raw,
    y,
    start_w=hc_w,
    maxiter=500,
)

add_candidate(
    name="nm_softmax_raw",
    method="nm",
    oof_pred=weighted_average(X_raw, nm_w),
    test_pred=weighted_average(T_raw, nm_w),
    weights=nm_w,
    params={
        "constraint": "softmax_sum1",
        "success": bool(nm_res.success),
        "message": str(nm_res.message),
        "nit": int(getattr(nm_res, "nit", -1)),
        "fun": float(nm_res.fun),
    },
    notes="Nelder-Mead on softmax weights; in-sample optimizer, interpret cautiously",
)

print("NM weights:")
for k, w in zip(model_keys, nm_w):
    print(k, w)


Nelder-Mead
Optimization terminated successfully.
         Current function value: -0.954294
         Iterations: 205
         Function evaluations: 475
nm_softmax_raw: 0.954294088
NM weights:
007 0.35368059905573773
008 0.13928860318843286
009 0.08828395759090685
012 0.04329992499221095
013 0.02688730359995934
014 0.10985831875858497
015 0.06979288103494631
016 0.11510313244669121
017 0.05380527933252969


In [15]:
# ============================================================
# Summary
# ============================================================

summary_df = pd.DataFrame(candidate_summary).sort_values("cv_auc", ascending=False)
display(summary_df)

summary_df.to_csv(CFG.OUTDIR / "blend_summary.csv", index=False)

# Expand weights table
weights_rows = []

for name, rec in candidate_records.items():
    if rec["weights"] is None:
        continue

    row = {
        "candidate": name,
        "method": rec["method"],
        "cv_auc": rec["score"],
    }

    for k, w in zip(model_keys, rec["weights"]):
        row[f"w_{k}"] = float(w)

    weights_rows.append(row)

weights_df = pd.DataFrame(weights_rows).sort_values("cv_auc", ascending=False)
display(weights_df)

weights_df.to_csv(CFG.OUTDIR / "blend_weights.csv", index=False)

,name,method,cv_auc,delta_vs_007,delta_vs_best_single,weights,params,notes,oof_min,oof_max,pred_min,pred_max
8,nm_softmax_raw,nm,0.954294,0.001024,0.001024,"[0.3536806, 0.1392886, 0.08828396, 0.04329992,...","{""constraint"": ""softmax_sum1"", ""success"": true...",Nelder-Mead on softmax weights; in-sample opti...,0.000055,0.996872,0.000062,0.996837
7,hc_nonnegative_raw,hc,0.954294,0.001024,0.001024,"[0.35173057, 0.13909413, 0.08873628, 0.0440160...","{""constraint"": ""nonnegative_sum1""}",greedy hill climb on raw OOF predictions; in-s...,0.000055,0.996860,0.000062,0.996814
6,logreg_meta_all,logreg,0.954266,0.000996,0.000996,None,"{""C"": 0.01219126212809401}",two-stage meta CV logistic regression on raw+r...,0.000035,0.990256,0.000035,0.989683
5,elasticnet_meta_all,elasticnet,0.954195,0.000925,0.000925,None,"{""alpha"": 0.12599210498948735, ""l1_ratio"": 0.1}",two-stage meta CV on raw+rank+logit features,0.001483,0.902807,0.001728,0.901490
3,avg_007_008_009_012_013_014_015_016_017,avg,0.954088,0.000818,0.000818,"[0.11111111, 0.11111111, 0.11111111, 0.1111111...",{},"simple average of ['007', '008', '009', '012',...",0.000065,0.996891,0.000086,0.996509
0,avg_007_008_012_013_014_015_016_017,avg,0.954060,0.000790,0.000790,"[0.125, 0.125, 0.0, 0.125, 0.125, 0.125, 0.125...",{},"simple average of ['007', '008', '012', '013',...",0.000063,0.996881,0.000090,0.996275
1,avg_007_009_012_013_014_015_016_017,avg,0.953944,0.000674,0.000674,"[0.125, 0.0, 0.125, 0.125, 0.125, 0.125, 0.125...",{},"simple average of ['007', '009', '012', '013',...",0.000071,0.996732,0.000094,0.996354
2,avg_008_009_012_013_014_015_016_017,avg,0.953895,0.000625,0.000625,"[0.0, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125...",{},"simple average of ['008', '009', '012', '013',...",0.000069,0.997090,0.000093,0.996364
4,ridge_meta_all,ridge,0.953666,0.000396,0.000396,None,"{""alpha"": 0.11746189430880191}",two-stage meta CV on raw+rank+logit features,-0.126359,1.019846,-0.097947,1.011785


,candidate,method,cv_auc,w_007,w_008,w_009,w_012,w_013,w_014,w_015,w_016,w_017
5,nm_softmax_raw,nm,0.954294,0.353681,0.139289,0.088284,0.043300,0.026887,0.109858,0.069793,0.115103,0.053805
4,hc_nonnegative_raw,hc,0.954294,0.351731,0.139094,0.088736,0.044016,0.023670,0.110799,0.074392,0.109024,0.058539
3,avg_007_008_009_012_013_014_015_016_017,avg,0.954088,0.111111,0.111111,0.111111,0.111111,0.111111,0.111111,0.111111,0.111111,0.111111
0,avg_007_008_012_013_014_015_016_017,avg,0.954060,0.125000,0.125000,0.000000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000
1,avg_007_009_012_013_014_015_016_017,avg,0.953944,0.125000,0.000000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000
2,avg_008_009_012_013_014_015_016_017,avg,0.953895,0.000000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000,0.125000


In [16]:
# ============================================================
# Save OOF / pred / submissions
# ============================================================

print("\n" + "=" * 80)
print("Save blend artifacts")
print("=" * 80)

submission_dir = CFG.OUTDIR / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)

target_col = [c for c in sub.columns if c != CFG.ID_COL][0]

for name, rec in candidate_records.items():
    safe_name = sanitize_name(name)

    oof_path = CFG.OUTDIR / f"oof_{CFG.EXP_ID}_{safe_name}.npy"
    pred_path = CFG.OUTDIR / f"pred_{CFG.EXP_ID}_{safe_name}.npy"
    sub_path = submission_dir / f"sub_{safe_name}_{CFG.EXP_ID}.csv"

    np.save(oof_path, rec["oof"])
    np.save(pred_path, rec["pred"])

    sub_out = sub.copy()
    sub_out[target_col] = safe_clip(rec["pred"])
    sub_out.to_csv(sub_path, index=False)

    print(name, rec["score"], sub_path)


Save blend artifacts
avg_007_008_012_013_014_015_016_017 0.9540600564642047 /kaggle/working/exp_20260504_010_blend_core_min/submissions/sub_avg_007_008_012_013_014_015_016_017_exp_20260504_010_blend_core_min.csv
avg_007_009_012_013_014_015_016_017 0.9539436848498155 /kaggle/working/exp_20260504_010_blend_core_min/submissions/sub_avg_007_009_012_013_014_015_016_017_exp_20260504_010_blend_core_min.csv
avg_008_009_012_013_014_015_016_017 0.9538948082548067 /kaggle/working/exp_20260504_010_blend_core_min/submissions/sub_avg_008_009_012_013_014_015_016_017_exp_20260504_010_blend_core_min.csv
avg_007_008_009_012_013_014_015_016_017 0.9540883725000631 /kaggle/working/exp_20260504_010_blend_core_min/submissions/sub_avg_007_008_009_012_013_014_015_016_017_exp_20260504_010_blend_core_min.csv
ridge_meta_all 0.9536662879259677 /kaggle/working/exp_20260504_010_blend_core_min/submissions/sub_ridge_meta_all_exp_20260504_010_blend_core_min.csv
elasticnet_meta_all 0.954194948917162 /kaggle/working/exp

In [17]:
# ============================================================
# Save result json
# ============================================================

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
        "created_at": "2026-05-04",
    },
    "inputs": {
        "model_specs": MODEL_SPECS,
        "model_keys": model_keys,
        "n_models": len(model_keys),
    },
    "individual": individual_df.to_dict(orient="records"),
    "correlation": {
        "pearson": pearson.to_dict(),
        "spearman": spearman_df.to_dict(),
    },
    "blend_summary": summary_df.to_dict(orient="records"),
    "blend_weights": weights_df.to_dict(orient="records"),
    "best": {
        "candidate": str(summary_df.iloc[0]["name"]),
        "method": str(summary_df.iloc[0]["method"]),
        "cv_auc": float(summary_df.iloc[0]["cv_auc"]),
    },
    "notes": [
        "This blend stage is intended to check whether 009 receives useful weight.",
        "Ridge/ElasticNet/LogReg use meta-CV on raw+rank+logit features.",
        "HC/NM optimize directly on full OOF and are more overfit-prone.",
        "If 009 receives meaningful weight, trying 009 + safe orig prior as next experiment is justified.",
        "If 009 receives little or zero weight, 009 + orig prior priority should be lowered.",
    ],
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "blend_summary": str(CFG.OUTDIR / "blend_summary.csv"),
        "blend_weights": str(CFG.OUTDIR / "blend_weights.csv"),
        "submissions_dir": str(submission_dir),
    },
}

with open(CFG.OUTDIR / "blend_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("\nSaved to:", CFG.OUTDIR)
print("Best candidate:")
print(summary_df.iloc[0].to_dict())


Saved to: /kaggle/working/exp_20260504_010_blend_core_min
Best candidate:
{'name': 'nm_softmax_raw', 'method': 'nm', 'cv_auc': 0.9542940876336243, 'delta_vs_007': 0.001023958732046082, 'delta_vs_best_single': 0.001023958732046082, 'weights': '[0.3536806, 0.1392886, 0.08828396, 0.04329992, 0.0268873, 0.10985832, 0.06979288, 0.11510313, 0.05380528]', 'params': '{"constraint": "softmax_sum1", "success": true, "message": "Optimization terminated successfully.", "nit": 205, "fun": -0.9542940876336243}', 'notes': 'Nelder-Mead on softmax weights; in-sample optimizer, interpret cautiously', 'oof_min': 5.498345153455961e-05, 'oof_max': 0.9968722993466009, 'pred_min': 6.189916140831915e-05, 'pred_max': 0.9968368733236812}
